In [0]:
# %sql
# drop table streaming_project.gold.driver_scd_details

In [0]:
%sql
CREATE TABLE IF NOT EXISTS streaming_project.gold.driver_scd_details
( 
  driver_name STRING,
  driver_nationality STRING,
  constructor_team STRING,
  driver_scd_start_year INT,
  driver_scd_end_year INT,
  driver_scd_status STRING
) USING DELTA
PARTITIONED BY (driver_name)

In [0]:
class Gold_driver_scd_details():
    main_path = "/Volumes/databricks_catalog/default/default_volume1"
    silver_path = "streaming_project/silver"
    gold_path = "streaming_project/gold"

    def __init__(self, table):
        self.table = table
    
    #reading
    def read_input(self):
      from pyspark.sql.functions import col,concat_ws,collect_set,row_number,element_at, size,expr,array_sort,explode,count,min
      from pyspark.sql.window import Window
      if spark.catalog.tableExists("streaming_project.gold.race_results"):
            Incr_race_results_df = (spark.read.table('streaming_project.gold.race_results'))

            Incr_race_results_distinct_df=(Incr_race_results_df
                                          .selectExpr("driver_name", "driver_nationality", "constructor_team", "race_year").distinct())

            # group different constructor teams in same year
            g_ct_df=(Incr_race_results_distinct_df
                  .groupBy("driver_name","driver_nationality","race_year")
                  .agg(concat_ws(',', collect_set(col('constructor_team'))).alias("constructor_team"))
                  .orderBy(col('driver_name'),col('race_year'))
                  .select('driver_name','driver_nationality','constructor_team','race_year')
                  )
            print('group different constructor teams in same year')
            display(g_ct_df.filter(col('driver_name').isin('Chris Amon','Jack Fairman','Jo Siffert')))

            # picking min race_year group by driver_name,constructor_team to one recored for each constructor_team if he play for many years to the same team
            w=Window.partitionBy(col('driver_name'),col('Constructor_team')).orderBy(col('Constructor_team'))
            main_df=(g_ct_df.withColumn('n_race_year',min(col('race_year')).over(w))
                  .filter(col('race_year')==col('n_race_year')).drop(col('n_race_year')))
            
            print('after working by picking min race year for each constructor team')
            display(main_df.filter(col('driver_name').isin('Chris Amon','Jack Fairman','Jo Siffert')))

      print("driver_scd_details:Incr_race_results_df batch count")
      display(main_df.select(count('*')))
      
      return main_df

    def apply_transformations(self,main_df):
        import time
        from pyspark.sql.functions import col,lit,row_number,max
        from pyspark.sql.window import Window
        if spark.catalog.tableExists("streaming_project.gold.driver_scd_details"):
                records_count=spark.sql("select count(*) as count from streaming_project.gold.driver_scd_details").collect()[0]['count']
                sample_df= (main_df
                    #.filter(col('driver_name').isin('Bob Bondurant','Alfonso Thiele','David Walker','Guy Ligier'))
                    .withColumn('rank',row_number().over(Window.partitionBy(col("driver_name")).orderBy(col("race_year"))))
                    )
                if records_count==0:
                    (sample_df
                    .withColumn('driver_scd_end_year',lit(None))
                    .withColumn('driver_scd_status',lit(True))
                    .filter((col('rank')==1))
                    .selectExpr(
                            "driver_name", "driver_nationality", "constructor_team", "race_year as driver_scd_start_year","driver_scd_end_year","driver_scd_status"
                            ).createOrReplaceTempView("initial_driver_scd_details"))
                    # printing intial loading data
                    print("initial_driver_scd_details")
                    display(spark.sql('''SELECT * FROM initial_driver_scd_details'''))

                    print(f"duplicated data if any in initial_driver_scd_details")
                    display(spark.sql('''
                                        SELECT driver_name,count(*) as count FROM initial_driver_scd_details
                                        group by driver_name
                                        having count(*)>1
                                        '''))
                    spark.sql('''
                            INSERT INTO streaming_project.gold.driver_scd_details
                            SELECT * FROM initial_driver_scd_details
                    ''')
                    print(f"initial load of driver_scd_details in driver_scd_details")
                    display(spark.sql('''
                                        SELECT * FROM streaming_project.gold.driver_scd_details
                                        '''))
                max_rank=(sample_df.agg(max(col("rank")).alias('max_rank'))).collect()[0]['max_rank']
                print(f"max_rank:{max_rank}")
                for r in range(2,int(max_rank)+1):
                    batch_df=(sample_df
                        .filter((col('rank')==r))
                        .orderBy(col('driver_name'),col('race_year'))
                        .selectExpr("driver_name", "driver_nationality", "constructor_team", "race_year"))
                    batch_df.createOrReplaceTempView("updated_driver_scd_details")
                
                    print(f"{r}:batch_df")
                    display(spark.sql('''SELECT * FROM updated_driver_scd_details'''))

                    print(f"duplicated data if any in updated_driver_scd_details")
                    display(spark.sql('''
                                SELECT driver_name,count(*) as count FROM updated_driver_scd_details
                                group by driver_name
                                having count(*)>1
                                '''))

                    # SCD Type 2 Implementation
                    spark.sql('''
                            MERGE INTO streaming_project.gold.driver_scd_details t
                            USING updated_driver_scd_details u
                            ON  (t.driver_name = u.driver_name AND t.driver_scd_status = TRUE)
                            WHEN MATCHED AND t.constructor_team != u.constructor_team THEN 
                            UPDATE SET 
                                t.driver_scd_end_year = u.race_year-1,
                                t.driver_scd_status = FALSE
                            WHEN NOT MATCHED THEN INSERT (
                                driver_name,driver_nationality,constructor_team,driver_scd_start_year,driver_scd_end_year,driver_scd_status
                            ) VALUES (
                                u.driver_name,u.driver_nationality,u.constructor_team,u.race_year,NULL,TRUE)
                    ''')
                    print("driver_scd_details with out appending new modified records")
                    display(spark.sql('''SELECT * FROM streaming_project.gold.driver_scd_details order by driver_name'''))

                    df1 = spark.sql('''
                                        select *  
                                        from streaming_project.gold.driver_scd_details where driver_scd_status=FALSE order by driver_name''')

                    
                    insert_df = (
                            batch_df.alias("src")
                            .join(df1.alias("tgt"), (col("src.driver_name")==col("tgt.driver_name"))
                                                    # (col("tgt.constructor_team")!=col("src.constructor_team"))&
                                                    # (col("tgt.driver_scd_end_year") == col("src.race_year")-1)
                                                    , "inner")
                            .select(
                                col("src.driver_name"),
                                col("src.driver_nationality"),
                                col("src.constructor_team"),
                                col("src.race_year").alias("driver_scd_start_year"),
                                lit(None).alias("driver_scd_end_year"),
                                lit(True).alias("driver_scd_status")
                            ).distinct()
                    )

                    insert_df.createOrReplaceTempView("insert_newly_modified")
                    spark.sql('''
                            INSERT INTO streaming_project.gold.driver_scd_details
                            SELECT * FROM insert_newly_modified
                    ''')
                    print("driver_scd_details with appending new modified records")
                    display(spark.sql('select * FROM streaming_project.gold.driver_scd_details order by driver_name'))
                    time.sleep(2)
        return spark.sql('select * FROM streaming_project.gold.driver_scd_details order by driver_name')
    
    def write_output(self,df):
        from pyspark.sql.functions import collect_set,col,concat_ws
        driver_ct_short_details= (df.orderBy(col('driver_name'),col('driver_scd_start_year'))
                          .select(col('driver_name'),col('constructor_team'))
                          .groupBy(col("driver_name"))
                          .agg(concat_ws(',', collect_set(col("constructor_team"))).alias("constructor_team"))
                          .select(col("driver_name"),col("constructor_team")))
        driver_ct_short_details.write.mode("overwrite").saveAsTable("streaming_project.gold.driver_ct_short_details")
    
    def process(self):
        print("Started gold-ingestion-driver_scd_details in runing....")
        main_df = self.read_input()  # return list of distinct race_years
        apply_tran_df = self.apply_transformations(main_df)  # return aggregated data
        self.write_output(apply_tran_df)  # write to gold table driver_ct_short_details

In [0]:
Gold_driver_scd_details_instance=Gold_driver_scd_details("driver_scd_details")
Gold_driver_scd_details_instance.process()
print("Successfully Gold_driver_scd_details is ran")

In [0]:
# from pyspark.sql.functions import collect_set,col,concat_ws
# df=spark.sql('select * FROM streaming_project.gold.driver_scd_details order by driver_name')
# display(df.orderBy(col('driver_name'),col('driver_scd_start_year')))
# driver_ct_short_details= (df.orderBy(col('driver_name'),col('driver_scd_start_year'))
#                           .select(col('driver_name'),col('constructor_team'))
#                           .groupBy(col("driver_name"))
#                           .agg(concat_ws(',', collect_set(col("constructor_team"))).alias("constructor_team"))
#                           .select(col("driver_name"),col("constructor_team")))
# display(driver_ct_short_details)
# driver_ct_short_details.write.mode("overwrite").saveAsTable("streaming_project.gold.driver_ct_short_details")

In [0]:
# def read_input():
#       from pyspark.sql.functions import col,concat_ws,collect_set,row_number,element_at, size,expr,array_sort,explode,count,min
#       from pyspark.sql.window import Window
#       if spark.catalog.tableExists("streaming_project.gold.race_results"):
#             Incr_race_results_df = (spark.read.table('streaming_project.gold.race_results'))

#             Incr_race_results_distinct_df=(Incr_race_results_df
#                                           .selectExpr("driver_name", "driver_nationality", "constructor_team", "race_year").distinct())

#             # group different constructor teams in same year
#             g_ct_df=(Incr_race_results_distinct_df
#                   .groupBy("driver_name","driver_nationality","race_year")
#                   .agg(concat_ws(',', collect_set(col('constructor_team'))).alias("constructor_team"))
#                   .orderBy(col('driver_name'),col('race_year'))
#                   .select('driver_name','driver_nationality','constructor_team','race_year')
#                   )
#             print('group different constructor teams in same year')
#             display(g_ct_df.filter(col('driver_name').isin('Chris Amon','Jack Fairman','Jo Siffert')))

#             # picking min race_year group by driver_name,constructor_team to one recored for each constructor_team if he play for many years to the same team
#             w=Window.partitionBy(col('driver_name'),col('Constructor_team')).orderBy(col('Constructor_team'))
#             main_df=(g_ct_df.withColumn('n_race_year',min(col('race_year')).over(w))
#                   .filter(col('race_year')==col('n_race_year')).drop(col('n_race_year')))
            
#             print('after working by picking min race year for each constructor team')
#             display(main_df.filter(col('driver_name').isin('Chris Amon','Jack Fairman','Jo Siffert')))

#       print("driver_scd_details:Incr_race_results_df batch count")
#       display(main_df.select(count('*')))
      
#       return main_df

# def apply_transformations(main_df):
#       from pyspark.sql.functions import col,lit,row_number,max
#       from pyspark.sql.window import Window
#       if spark.catalog.tableExists("streaming_project.gold.driver_scd_details"):
#             records_count=spark.sql("select count(*) as count from streaming_project.gold.driver_scd_details").collect()[0]['count']
#             sample_df= (main_df
#                   #.filter(col('driver_name').isin('Chris Amon','Jack Fairman','Jo Siffert'))
#                   .withColumn('rank',row_number().over(Window.partitionBy(col("driver_name")).orderBy(col("race_year"))))
#                   )
#             if records_count==0:
#                   (sample_df
#                   .withColumn('driver_scd_end_year',lit(None))
#                   .withColumn('driver_scd_status',lit(True))
#                   .filter((col('rank')==1))
#                   .selectExpr(
#                         "driver_name", "driver_nationality", "constructor_team", "race_year as driver_scd_start_year","driver_scd_end_year","driver_scd_status"
#                         ).createOrReplaceTempView("initial_driver_scd_details"))
#                   # printing intial loading data
#                   print("initial_driver_scd_details")
#                   display(spark.sql('''SELECT * FROM initial_driver_scd_details'''))

#                   print(f"duplicated data if any in initial_driver_scd_details")
#                   display(spark.sql('''
#                                     SELECT driver_name,count(*) as count FROM initial_driver_scd_details
#                                     group by driver_name
#                                     having count(*)>1
#                                     '''))
#                   spark.sql('''
#                         INSERT INTO streaming_project.gold.driver_scd_details
#                         SELECT * FROM initial_driver_scd_details
#                   ''')
#                   print(f"initial load of driver_scd_details in driver_scd_details")
#                   display(spark.sql('''
#                                     SELECT * FROM streaming_project.gold.driver_scd_details
#                                     '''))
#             max_rank=(sample_df.agg(max(col("rank")).alias('max_rank'))).collect()[0]['max_rank']
#             print(f"max_rank:{max_rank}")
#             for r in range(2,int(max_rank)+1):
#                   batch_df=(sample_df
#                       .filter((col('rank')==r))
#                       .orderBy(col('driver_name'),col('race_year'))
#                       .selectExpr("driver_name", "driver_nationality", "constructor_team", "race_year"))
#                   batch_df.createOrReplaceTempView("updated_driver_scd_details")
            
#                   print(f"{r}:batch_df")
#                   display(spark.sql('''SELECT * FROM updated_driver_scd_details'''))

#                   print(f"duplicated data if any in updated_driver_scd_details")
#                   display(spark.sql('''
#                               SELECT driver_name,count(*) as count FROM updated_driver_scd_details
#                               group by driver_name
#                               having count(*)>1
#                               '''))

#                   # SCD Type 2 Implementation
#                   spark.sql('''
#                         MERGE INTO streaming_project.gold.driver_scd_details t
#                         USING updated_driver_scd_details u
#                         ON  (t.driver_name = u.driver_name AND t.driver_scd_status = TRUE)
#                         WHEN MATCHED AND t.constructor_team != u.constructor_team THEN 
#                         UPDATE SET 
#                               t.driver_scd_end_year = u.race_year-1,
#                               t.driver_scd_status = FALSE
#                         WHEN NOT MATCHED THEN INSERT (
#                               driver_name,driver_nationality,constructor_team,driver_scd_start_year,driver_scd_end_year,driver_scd_status
#                         ) VALUES (
#                               u.driver_name,u.driver_nationality,u.constructor_team,u.race_year,NULL,TRUE)
#                   ''')
#                   print("driver_scd_details with out appending new modified records")
#                   display(spark.sql('''SELECT * FROM streaming_project.gold.driver_scd_details order by driver_name'''))

#                   df1 = spark.sql('''
#                                     select *  
#                                     from streaming_project.gold.driver_scd_details where driver_scd_status=FALSE order by driver_name''')

                  
#                   insert_df = (
#                         batch_df.alias("src")
#                         .join(df1.alias("tgt"), (col("src.driver_name")==col("tgt.driver_name"))
#                                                 # (col("tgt.constructor_team")!=col("src.constructor_team"))&
#                                                 # (col("tgt.driver_scd_end_year") == col("src.race_year")-1)
#                                                 , "inner")
#                         .select(
#                               col("src.driver_name"),
#                               col("src.driver_nationality"),
#                               col("src.constructor_team"),
#                               col("src.race_year").alias("driver_scd_start_year"),
#                               lit(None).alias("driver_scd_end_year"),
#                               lit(True).alias("driver_scd_status")
#                            ).distinct()
#                   )

#                   insert_df.createOrReplaceTempView("insert_newly_modified")
#                   spark.sql('''
#                         INSERT INTO streaming_project.gold.driver_scd_details
#                         SELECT * FROM insert_newly_modified
#                   ''')
#                   print("driver_scd_details with appending new modified records")
#                   display(spark.sql('select * FROM streaming_project.gold.driver_scd_details order by driver_name'))
                  

In [0]:
# main_df=read_input()
# apply_transformations(main_df)
# sample_df= (main_df.filter(col('driver_name').isin('Larry Perkins','Lewis Hamilton'))
#                   .withColumn('rank',row_number().over(Window.partitionBy(col("driver_name")).orderBy(col("race_year"))))
#                   )
# display(sample_df)
# Identify when constructor_team changes
# df = (df.filter(col('driver_name').isin('Larry Perkins','Lewis Hamilton'))
#       .groupBy(col('driver_name'),col('driver_nationality'),col('constructor_team'))
#       .agg(collect_set(col('race_year')).alias("race_year_list"))
#       .withColumn('race_year_list', array_sort(col('race_year_list')))
#       .withColumn("m_race_year_list",
#         # element_at is 1-based, so first element is 1, last is size
#         expr("array(element_at(race_year_list, 1), element_at(race_year_list, size(race_year_list)) + 1)")
#       )
#       .withColumn("race_year",explode(col("m_race_year_list")))
#       #.drop(col('m_race_year_list'),col('race_year_list'))
#       )